In [1]:
import os
import torch
import math
import time
import inspect
import numpy as np
import torch.nn as nn
import urllib.request
import torch.nn.functional as F
from typing import Optional
from dataclasses import dataclass
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import ReduceLROnPlateau

from tqdm.notebook import tqdm

In [2]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
device_type = "cuda" if device.type.startswith("cuda") else "cpu"

In [3]:
@dataclass
class CFG:
    experiment_name: str = "tiny_shakespeare"
    max_seq_len: int = 256
    vocab_size: int = 65
    dim: int = 384
    n_layers: int = 6
    n_heads_q: int = 6
    n_heads_kv: Optional[int] = None
    dropout: float = 0.02

    multiple_of: int = 256
    ffn_multiplier: Optional[int] = None
    norm_eps: float = 1e-6
    init_std: float = 0.02

    state_dir: str = "model_states"
    grad_clip: float = 1.0
    grad_accumulation_steps: int = 1
    batch_size_train: int = 64
    batch_size_valid: int = 64
    num_workers: int = 4
    valid_interval: int = 1
    n_epochs: int = 10
    last_state: Optional[str] = None
    best_state_file_name: str = "best_model"
    last_state_file_name: str = "last_model"

    init_lr: float = 1e-3
    min_lr: float = 1e-4
    warmup_steps: int = 100
    max_steps: int = 2500
    valid_every_steps: int = 250
    valid_batches: int = 25
    weight_decay: float = 0.1
    betas: tuple = (0.9, 0.95)

args = CFG()

In [4]:
class InputEmbeddings(nn.Module):
    def __init__(self, embed_dim: int, vocab_size: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.vocab_size  = vocab_size
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        
    def forward(self, x):
        return self.embeddings(x)


In [5]:
class RPositionalEmbeddings(nn.Module):
    def __init__(self, head_dim: int, seq_len: int, base: float= 10000.0):
        super().__init__()
        assert head_dim%2==0, "Head dim must be even."
        self.head_dim = head_dim
        self.seq_len = seq_len
        self.base = base

        pow = -2*(torch.arange(0, head_dim//2).float())/head_dim
        theta = (base**pow)
        m = torch.arange(seq_len).float()
        #multiply each m with all theta
        angles = torch.outer(m, theta) #seq_len,head dim/2

        full_angle = torch.concat([angles, angles], dim=1) #seq len, head dim
        self.register_buffer("cos_table", torch.cos(full_angle), persistent=False)
        self.register_buffer("sin_table", torch.sin(full_angle), persistent=False)

    def forward(self, x: torch.Tensor, positions: torch.Tensor):
        cos = self.cos_table[positions].unsqueeze(1) # batch size, 1, seq len, head dim
        sin = self.sin_table[positions].unsqueeze(1)
        # bs, heads, seq len, head dim
        out = (x*cos + torch.concat([-x[:,:,:,self.head_dim//2:], x[:,:,:,:self.head_dim//2]],\
                                     dim=-1)*sin).type_as(x)
        return out


In [6]:
r = RPositionalEmbeddings(16,10)

x = torch.randn(8,4,10, 16) #bs, heads, seq len, head dim
pos = torch.arange(10).repeat(8,1)

print(torch.norm(x))

# if implementation is correct, the norms are preserved for rotation.
print(torch.norm(r(x,pos)))

tensor(70.3392)
tensor(70.3392)


In [7]:
r1= RPositionalEmbeddings(16, 256)
q = torch.randn(1,1,1,16); k = torch.randn(1,1,1,16)

def dot(qp, kp):
    a = r1(q, torch.tensor([[qp]])); b = r1(k, torch.tensor([[kp]]))
    return (a*b).sum().item()

#Rotate q at position 5 and k at position 8, take the dot product.
#Now rotate q at 105 and k at 108 and take the dot product. They must be equal.
print(f"pos(5,8)     -> {dot(5,8):.6f}")
print(f"pos(105,108) -> {dot(105,108):.6f}")
print(f"pos(5,9)     -> {dot(5,9):.6f}   (different gap, should differ)")

pos(5,8)     -> -2.878451
pos(105,108) -> -2.878451
pos(5,9)     -> -2.169086   (different gap, should differ)


In [8]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, dim: int, h_q: int, h_kv: int, dropout: float, rope: RPositionalEmbeddings):
        super().__init__()
        self.dim = dim
        self.h_q = h_q
        self.h_kv = h_kv
        self.repeats = h_q//h_kv
        self.positional_embed = rope
        
        self.d_k = dim//h_q
        self.w_q = nn.Linear(dim,self.d_k*h_q, bias= False)
        self.w_k = nn.Linear(dim,self.d_k*h_kv, bias= False)        
        self.w_v = nn.Linear(dim, self.d_k*h_kv, bias= False)  
        self.w_o = nn.Linear(dim, dim, bias= False)  
        self.dropout = nn.Dropout(dropout) 
        
    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        scores = query@key.transpose(-2,-1)/math.sqrt(d_k)

        if mask is not None:
            scores.masked_fill_(mask==0, -1e9)
        scores = scores.softmax(dim=-1)
        
        ### do we need ?
        if dropout is not None:
            scores = dropout(scores)
            
        return scores@value,scores 
        
    def forward(self, q, k, v, pos,mask):
        # batch, seq len, embed dim -> batch, seq_len, d_k*h_q
        query = self.w_q(q)
        # batch, seq len, embed dim -> batch, seq_len, d_k*h_kv
        key = self.w_k(k)
        value = self.w_v(v)
        
        # batch, h_q, seq, d_k
        query = query.view(query.shape[0], query.shape[1], self.h_q, self.d_k).transpose(1,2)
        # batch, h_kv, seq, d_k
        key = key.view(key.shape[0], key.shape[1], self.h_kv, self.d_k).transpose(1,2)
        value = value.view(value.shape[0], value.shape[1], self.h_kv, self.d_k).transpose(1,2)
        #apply positional embeddings to query and key
        query = self.positional_embed(query, pos)
        key = self.positional_embed(key, pos)
        if self.repeats!=1:
            # batch, h_q, seq, d_k
            key = key.repeat_interleave(repeats=self.repeats, dim=1)
            value = value.repeat_interleave(repeats=self.repeats, dim=1)

        x = F.scaled_dot_product_attention(query, key, value, attn_mask=mask,\
                                        dropout_p=self.dropout.p if self.training else 0.0, is_causal=mask is None)
        x = x.transpose(1,2).reshape(x.shape[0],-1, self.dim)
        return self.w_o(x)
        

In [9]:
rope = RPositionalEmbeddings(512//32, 16)
mha = MultiHeadAttention(512, 32, 8, 0.0, rope)#embed_dim, heads query, heads key/value, dropout, rot embed  

In [10]:
x = torch.randn(4,16,512)
pos = torch.arange(16).repeat(4,1)
out = mha(x,x,x, pos,None)
out.shape

torch.Size([4, 16, 512])

In [11]:
class RMSNorm(nn.Module):

    def __init__(self, dim ,eps:float = 1e-6):
        super().__init__()
        self.dim = dim
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        v = x.float()
        v = v*torch.rsqrt((v**2).mean(dim=-1, keepdim=True)+self.eps)
        v = v*self.weight.float()

        return v.type_as(x)

In [12]:
class FFN(nn.Module):
    def __init__(self, dim: int, ffn_multipler: Optional[int], multiple_of: int) -> None:
        super().__init__()
        hidden_dim = 4*dim
        hidden_dim = int(2*hidden_dim/3)
        if ffn_multipler is not None:
            hidden_dim = int(hidden_dim*ffn_multipler)
        #next multiple of multiple_of
        hidden_dim = multiple_of*((hidden_dim+multiple_of-1)//multiple_of)
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)

    def forward(self,x):
        swish = F.silu(self.w1(x))
        x_V = self.w3(x)
        x = swish*x_V
        x = self.w2(x)

        return x


In [13]:
class EncoderBlock(nn.Module):
    def __init__(self, args: CFG, rope: RPositionalEmbeddings):
        super().__init__()
        self.dim = args.dim
        self.n_heads_q = args.n_heads_q
        self.n_heads_kv = args.n_heads_kv if args.n_heads_kv is not None else args.n_heads_q
        assert self.n_heads_q % self.n_heads_kv == 0, "n_heads_q must be divisible by n_heads_kv"
        self.head_dim = args.dim // self.n_heads_q
        self.attention = MultiHeadAttention(self.dim, self.n_heads_q, self.n_heads_kv, args.dropout, rope)

        self.ffn = FFN(self.dim, ffn_multipler=args.ffn_multiplier, multiple_of=args.multiple_of)
        self.norm1 = RMSNorm(self.dim, args.norm_eps)
        self.norm2 = RMSNorm(self.dim, args.norm_eps)
        self.dropout = nn.Dropout(args.dropout)

    def forward(self, x, pos, mask):
        # Attention block
        h = self.norm1(x)
        attn_out = self.attention(h, h, h, pos, mask)
        x = x + self.dropout(attn_out)

        # Feedforward block
        ffn_out = self.ffn(self.norm2(x))
        x = x + self.dropout(ffn_out)

        return x

In [14]:
class Transformer(nn.Module):
    def __init__(self, args: CFG):
        super().__init__()
        self.args = args
        self.encoder_blocks = nn.ModuleList()
        rope = RPositionalEmbeddings(args.dim//args.n_heads_q, args.max_seq_len)
        for _ in range(args.n_layers):
            self.encoder_blocks.append(EncoderBlock(args, rope))
        self.embeddings = InputEmbeddings(args.dim, args.vocab_size)
        self.projection = nn.Linear(args.dim, args.vocab_size, bias=False)
        self.norm = RMSNorm(args.dim, args.norm_eps)

        self._init_weights(args.init_std)

    def forward(self, x, pos, mask):
        assert pos.max() < self.args.max_seq_len, "Position indices exceed maximum sequence length."
        x = self.embeddings(x)
        for block in self.encoder_blocks:
            x = block(x, pos, mask)
        x = self.norm(x)
        x = self.projection(x)
        return x

    def _init_weights(self, std):
        # every projection and the embedding table: normal(0, std).
        # RMSNorm gains are left at ones - matching on module type avoids them.
        for mod in self.modules():
            if isinstance(mod, nn.Linear):
                nn.init.normal_(mod.weight, mean=0.0, std=std)
                if mod.bias is not None:
                    nn.init.zeros_(mod.bias)
            elif isinstance(mod, nn.Embedding):
                nn.init.normal_(mod.weight, mean=0.0, std=std)

        # each block writes to the residual stream twice (w_o and w2), so its
        # variance would grow like sqrt(2*n_layers) with depth. Scaling the two
        # residual-writing projections by the inverse keeps it depth-independent.
        scale = 1.0/math.sqrt(2*len(self.encoder_blocks))
        for block in self.encoder_blocks:
            block.attention.w_o.weight.data.mul_(scale)
            block.ffn.w2.weight.data.mul_(scale)


In [15]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self,full_text: str, text: str, block_size: int):
        self.block_size = block_size
        self.text = text
        self.full_text = full_text
        print(f"Length of dataset in characters: {len(self.text):,}")
        self.vocab = sorted(list(set(self.full_text)))
        self.vocab_size = len(self.vocab)
        print(f"Unique characters in dataset: {self.vocab_size:,}")
        self.vocab_to_idx = {ch: i for i, ch in enumerate(self.vocab)}
        self.idx_to_vocab = {i: ch for i, ch in enumerate(self.vocab)}
        self.text_tokens = torch.tensor([self.vocab_to_idx[c] for c in self.text], dtype=torch.long)

    def __len__(self):
        return len(self.text_tokens) - self.block_size

    def __getitem__(self, idx):
        x = self.text_tokens[idx:idx+self.block_size]
        y = self.text_tokens[idx+1:idx+self.block_size+1]
        return x, y

In [16]:
def save_model(model, optimizer, step, device, state_file_name, state_dir, loss):
    
    if not os.path.exists(state_dir):
        os.makedirs(state_dir)

    if 'best' in state_file_name:
        state_path = os.path.join(state_dir, state_file_name + '.pth')
    else:   
        state_path = os.path.join(state_dir, state_file_name + '_' + str(step)+ '.pth')

    # make sure you transfer the model to cpu.
    if device == 'cpu':
        model.to('cpu')

    # save the state_dict
    torch.save({
        'model_state_dict': model._orig_mod.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'step': torch.tensor(step),
        'loss': torch.tensor(loss)
    }, state_path)
    
    #transfer model to gpu again
    if device == 'cuda':
        model.to('cuda')
    
    return

In [17]:
def load_model(model, state_file_name ,state_dir=args.state_dir):
    state_path = os.path.join(state_dir, state_file_name)

    # loading the model and getting model parameters by using load_state_dict
    state = torch.load(state_path)
    model._orig_mod.load_state_dict(state['model_state_dict'])

    return model

In [18]:
def lr_at(step: int, cfg: CFG) -> float:
    """Linear warmup for cfg.warmup_steps, then cosine decay to cfg.min_lr."""
    if step < cfg.warmup_steps:
        return cfg.init_lr*(step+1)/cfg.warmup_steps
    if step >= cfg.max_steps:
        return cfg.min_lr
    progress = (step - cfg.warmup_steps)/max(1, cfg.max_steps - cfg.warmup_steps)
    return cfg.min_lr + 0.5*(cfg.init_lr - cfg.min_lr)*(1.0 + math.cos(math.pi*progress))


In [19]:
def train(
    train_config: CFG,
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    train_loader: torch.utils.data.DataLoader,
    valid_loader: torch.utils.data.DataLoader,
    epoch_idx: int,
    loss_fn: torch.nn.modules.loss.CrossEntropyLoss,
    vocab_size: int,
    device: torch.device,
    summary_writer: SummaryWriter,
    global_step: int,
    best_loss: float
    ):

    batch_loss = np.array([])
    norms = np.array([])
    model.train()
    batch_iterator = tqdm(train_loader, desc=f"Processing Epoch {epoch_idx}")
    t0 = time.time()
    tokens_processed = 0
    for step,(x, y) in enumerate(batch_iterator):
        pos = torch.arange(x.shape[1], device=device).repeat(x.shape[0], 1)
        inp = x.to(device, non_blocking=True)
        labels = y.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            out = model(inp, pos, mask=None)
            loss = loss_fn(out.view(-1, vocab_size),labels.view(-1))

        batch_loss = np.append(batch_loss, [loss.item()])
        tokens_processed += inp.numel()
        loss = loss / train_config.grad_accumulation_steps
        loss.backward()

        if (step+1)%train_config.grad_accumulation_steps == 0:
            # warmup + cosine, applied per optimizer step
            lr = lr_at(global_step, train_config)
            for group in optimizer.param_groups:
                group['lr'] = lr

            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), train_config.grad_clip)
            norms = np.append(norms, [norm.item()])
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

            # step-based validation
            if global_step % train_config.valid_every_steps == 0:
                valid_loss = valid(train_config, model, valid_loader, epoch_idx, loss_fn,
                                   vocab_size, device, max_batches=train_config.valid_batches)
                window = batch_loss[-train_config.valid_every_steps:].mean()
                note = ''
                if valid_loss < best_loss:
                    best_loss = valid_loss
                    note = '   <- best, saving'
                    save_model(model, optimizer, global_step, device=device,
                               state_file_name=train_config.best_state_file_name,
                               state_dir=train_config.state_dir, loss=best_loss)
                print('step {:>6}  lr {:.2e}  train {:.4f}  valid {:.4f}{}'.format(
                    global_step, lr, window, valid_loss, note))
                summary_writer.add_scalar('Loss/Valid', valid_loss, global_step)
                summary_writer.add_scalar('Loss/TrainWindow', window, global_step)
                summary_writer.add_scalar('Learning Rate', lr, global_step)

            if global_step >= train_config.max_steps:
                break

        batch_iterator.set_postfix({"loss": f"{loss.item()*train_config.grad_accumulation_steps:.4f}"})

    dt = time.time() - t0
    tokens_per_sec = tokens_processed/dt
    epoch_loss = batch_loss.mean()
    epoch_norm = norms.mean() if len(norms) else float('nan')
    print('Epoch: {} Train Loss: {:.3f} Average Norm: {:.3f} Tokens/sec: {:.2f} Time: {:.2f}s'.format(
        epoch_idx, epoch_loss, epoch_norm, tokens_per_sec, dt))
    return epoch_loss, global_step, best_loss


In [20]:
def valid(
    train_config: CFG,
    model: Transformer,
    valid_loader: torch.utils.data.DataLoader,
    epoch_idx: int,
    loss_fn: torch.nn.modules.loss.CrossEntropyLoss,
    vocab_size: int,
    device: torch.device,
    max_batches: Optional[int] = None
    )-> float:

    batch_loss = np.array([])
    model.eval()
    batch_iterator = valid_loader
    with torch.no_grad():
        for step,(x, y) in enumerate(batch_iterator):
            pos = torch.arange(x.shape[1], device=device).repeat(x.shape[0], 1)
            inp = x.to(device)
            labels = y.to(device)

            with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
                out = model(inp, pos, mask=None)
                loss = loss_fn(out.view(-1, vocab_size),labels.view(-1))

            batch_loss = np.append(batch_loss, [loss.item()])
            if max_batches is not None and step+1 >= max_batches:
                break
    epoch_loss = batch_loss.mean()
    return epoch_loss

In [21]:
def main(
    args: CFG,
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    dataset_train: torch.utils.data.Dataset,
    dataset_valid: torch.utils.data.Dataset,
    scheduler,                      # unused: the cosine schedule is applied per step inside train()
    summary_writer: SummaryWriter,
    device: torch.device
    ):

    train_loader = DataLoader(dataset_train, batch_size=args.batch_size_train,
                              num_workers=args.num_workers, shuffle=True,
                              drop_last=True, pin_memory=True)
    valid_loader = DataLoader(dataset_valid, batch_size=args.batch_size_valid,
                              num_workers=2, pin_memory=True)
    loss_fn = nn.CrossEntropyLoss()
    vocab_size = args.vocab_size
    model.to(device)

    best_loss = float(np.inf)
    global_step = 0
    if args.last_state is not None:
        state_path = os.path.join(args.state_dir, args.last_state)
        state = torch.load(state_path)
        model._orig_mod.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        best_loss = state['loss'].item()
        global_step = state['step'].item()
        print(f"Loaded model from {args.last_state}")
        print(f"Best Loss: {best_loss:.4f}, Resuming from step {global_step}")

    print('warmup {} steps, cosine {:.1e} -> {:.1e} over {} steps'.format(
        args.warmup_steps, args.init_lr, args.min_lr, args.max_steps))
    print('validating every {} steps on {} batches\n'.format(
        args.valid_every_steps, args.valid_batches))

    t_begin = time.time()
    for epoch in range(args.n_epochs):
        train_loss, global_step, best_loss = train(
            args, model, optimizer, train_loader, valid_loader, epoch, loss_fn,
            vocab_size, device, summary_writer, global_step, best_loss)
        summary_writer.add_scalar('Loss/Train', train_loss, epoch)
        if global_step >= args.max_steps:
            print('Reached max_steps ({}).'.format(args.max_steps))
            break

    save_model(model, optimizer, global_step, device=device,
               state_file_name=args.last_state_file_name,
               state_dir=args.state_dir, loss=best_loss)
    print("Total time: {:.2f}, Best Valid Loss: {:.3f}".format(time.time() - t_begin, best_loss))


In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("./data/input.txt"):
    urllib.request.urlretrieve(url, "./data/input.txt")
text = open("../data/input.txt", "r").read()

d_train = Dataset(full_text=text, text=text[:int(len(text)*0.9)], block_size=args.max_seq_len)
d_val = Dataset(full_text=text, text=text[int(len(text)*0.9):], block_size=args.max_seq_len)

Length of dataset in characters: 1,003,854
Unique characters in dataset: 65
Length of dataset in characters: 111,540
Unique characters in dataset: 65


In [23]:
model = Transformer(args)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params/1e6:.2f}M")

Trainable parameters: 10.67M


In [24]:
def get_optimizer_params(model, lr, weight_decay=0.0):
    no_decay = ["bias", "norm"]
    optimizer_parameters = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'lr': lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'lr': lr, 'weight_decay': 0.0}
    ]
    return optimizer_parameters


In [25]:
optimizer_params = get_optimizer_params(model, args.init_lr, args.weight_decay)
fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
use_fused = fused_available and device_type == "cuda" 

optimizer = torch.optim.AdamW(
    optimizer_params,
    lr = args.init_lr,
    betas=args.betas,
    fused=use_fused
)

schedular = ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.3, patience = 2, min_lr= 1e-6)

In [26]:
model.to(device)
model = torch.compile(model)

In [27]:
summary_writer = SummaryWriter(log_dir=args.experiment_name)

In [28]:
%%time
main(args, model, optimizer, d_train, d_val, schedular, summary_writer, device)

warmup 100 steps, cosine 1.0e-03 -> 1.0e-04 over 2500 steps
validating every 250 steps on 25 batches



Processing Epoch 0:   0%|          | 0/15681 [00:00<?, ?it/s]

step    250  lr 9.91e-04  train 1.9102  valid 1.3919   <- best, saving
step    500  lr 9.40e-04  train 1.2700  valid 1.2520   <- best, saving
step    750  lr 8.47e-04  train 1.0931  valid 1.2797
step   1000  lr 7.23e-04  train 0.8820  valid 1.4424
step   1250  lr 5.80e-04  train 0.6192  valid 1.7871
step   1500  lr 4.34e-04  train 0.3877  valid 2.2467
step   1750  lr 3.00e-04  train 0.2388  valid 2.5814
step   2000  lr 1.93e-04  train 0.1607  valid 2.9258
step   2250  lr 1.24e-04  train 0.1252  valid 3.1189
step   2500  lr 1.00e-04  train 0.1097  valid 3.2388
Epoch: 0 Train Loss: 0.680 Average Norm: 0.581 Tokens/sec: 453393.92 Time: 90.34s
Reached max_steps (2500).
Total time: 90.47, Best Valid Loss: 1.252
CPU times: user 1min 30s, sys: 4.6 s, total: 1min 35s
Wall time: 1min 30s
